# Análise Exploratória de Dados (EDA) — House Prices (Ames, Iowa)

Este notebook apresenta uma **Análise Exploratória de Dados (EDA) completa e documentada** do dataset *House Prices: Advanced Regression Techniques*, cujo objetivo final é prever `SalePrice` (preço de venda) de imóveis residenciais na cidade de Ames, Iowa (EUA), entre 2006 e 2010.

O foco aqui **não é construir o modelo preditivo**, e sim entender profundamente os dados: sua estrutura, qualidade, distribuições, relações entre variáveis e padrões que devem orientar as decisões de limpeza, tratamento e engenharia de atributos em uma etapa posterior de modelagem.

### Estrutura do notebook

1. Configuração do ambiente e carregamento dos dados
2. Visão geral do dataset (estrutura, tipos, dimensões)
3. Análise da variável-alvo (`SalePrice`)
4. Análise de dados ausentes
5. Análise univariada das variáveis numéricas
6. Análise univariada das variáveis categóricas
7. Análise bivariada: variáveis numéricas x `SalePrice`
8. Análise bivariada: variáveis categóricas x `SalePrice`
9. Análise de correlação (multivariada)
10. Detecção de outliers e pontos de atenção
11. Análise temporal (ano de construção, venda, reforma)
12. Síntese dos principais insights e recomendações para modelagem


## 1. Configuração do ambiente e carregamento dos dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 100
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 140)

PALETTE = 'viridis'


In [ ]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

print(f'Dimensões do treino: {train.shape[0]} linhas x {train.shape[1]} colunas')
print(f'Dimensões do teste : {test.shape[0]} linhas x {test.shape[1]} colunas')
print(f'\nO conjunto de teste não possui a coluna alvo (SalePrice), que é o que queremos prever.')

train.head()


> **Nota metodológica:** a EDA a seguir é conduzida **exclusivamente sobre o conjunto de treino** (`train.csv`), pois é o único que contém a variável-alvo `SalePrice`. Essa é uma boa prática para evitar qualquer viés proveniente da observação do conjunto de teste (*data leakage* / *snooping*) antes da fase de modelagem.

## 2. Visão geral do dataset

Antes de qualquer análise específica, é importante entender a "anatomia" do dataset: quantas variáveis existem, de que tipo, e como estão organizadas conceitualmente.

In [ ]:
train.info()


In [ ]:
dtype_counts = train.dtypes.astype(str).replace({'str':'object'}).value_counts()
print('Contagem de colunas por tipo de dado:')
print(dtype_counts)

plt.figure(figsize=(5,4))
dtype_counts.plot(kind='bar', color='slateblue')
plt.title('Quantidade de colunas por tipo de dado')
plt.ylabel('Número de colunas')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()


O dataset possui **81 colunas** (incluindo `Id` e o alvo `SalePrice`): a maioria numérica (inteiros/floats representando áreas, contagens e anos) e **43 colunas categóricas** (texto), que descrevem qualidade, tipo, estilo e características construtivas do imóvel.

### 2.1 Estatísticas descritivas

Uma primeira leitura das variáveis numéricas já revela escalas muito diferentes (de `PoolArea`, quase sempre zero, até `LotArea`, em dezenas de milhares de pés quadrados) — algo que precisará ser tratado antes de qualquer modelo baseado em distância.

In [ ]:
train.describe().T.style.background_gradient(cmap='Blues', axis=0)


In [ ]:
cat_cols = train.select_dtypes(include=['object']).columns.tolist()
num_cols = train.select_dtypes(include=['int64','float64']).columns.tolist()
num_cols_no_id = [c for c in num_cols if c not in ['Id']]

print(f'Variáveis categóricas ({len(cat_cols)}):')
print(cat_cols)
print(f'\nVariáveis numéricas ({len(num_cols_no_id)}, excluindo Id):')
print(num_cols_no_id)


In [ ]:
train[cat_cols].describe().T


Estatísticas descritivas das variáveis categóricas: `count` (não nulos), `unique` (número de categorias distintas), `top` (categoria mais frequente) e `freq` (frequência dessa categoria). Note que várias variáveis são fortemente dominadas por uma única categoria (ex.: `Utilities`, `Street`), o que sugere baixo poder discriminativo — um ponto para revisitar na modelagem.

## 3. Análise da variável-alvo (`SalePrice`)

Como `SalePrice` é o que queremos prever, entender seu comportamento é o ponto de partida mais importante da EDA.

In [ ]:
print(train['SalePrice'].describe().apply(lambda x: f'{x:,.2f}'))
print(f"\nAssimetria (skewness) : {train['SalePrice'].skew():.3f}")
print(f"Curtose (kurtosis)    : {train['SalePrice'].kurt():.3f}")


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16,4.5))

sns.histplot(train['SalePrice'], kde=True, ax=axes[0], color='royalblue', bins=40)
axes[0].set_title('Distribuição de SalePrice')
axes[0].xaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[0].xaxis.set_major_locator(mticker.MaxNLocator(nbins=5))
axes[0].tick_params(axis='x', rotation=30)

sns.boxplot(x=train['SalePrice'], ax=axes[1], color='lightcoral')
axes[1].set_title('Boxplot de SalePrice')
axes[1].xaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
axes[1].xaxis.set_major_locator(mticker.MaxNLocator(nbins=5))
axes[1].tick_params(axis='x', rotation=30)

stats.probplot(train['SalePrice'], plot=axes[2])
axes[2].set_title('Q-Q Plot (normalidade)')

plt.tight_layout()
plt.show()


A distribuição de `SalePrice` é **assimetria à direita** (skew ≈ 1.88): a maioria das casas se concentra na faixa de US\$130 mil–215 mil, mas existe uma cauda longa de imóveis de altíssimo valor que puxa a média (US\$180.921) para cima da mediana (US\$163.000). O *Q-Q plot* confirma o desvio em relação à normalidade, especialmente na cauda superior.

Como a métrica oficial da competição usa o **logaritmo** do preço, aplicamos `log1p(SalePrice)` para avaliar se essa transformação aproxima a distribuição de uma normal — o que tende a beneficiar bastante modelos lineares.

In [ ]:
log_price = np.log1p(train['SalePrice'])

fig, axes = plt.subplots(1, 2, figsize=(11,4.5))
sns.histplot(log_price, kde=True, ax=axes[0], color='darkorange', bins=40)
axes[0].set_title('Distribuição de log1p(SalePrice)')

stats.probplot(log_price, plot=axes[1])
axes[1].set_title('Q-Q Plot de log1p(SalePrice)')
plt.tight_layout()
plt.show()

print(f'Assimetria original : {train["SalePrice"].skew():.3f}')
print(f'Assimetria log1p    : {log_price.skew():.3f}')


**Conclusão:** a transformação logarítmica reduz drasticamente a assimetria (de ~1.88 para ~0.12) e deixa a distribuição muito mais próxima da normal, confirmando que **treinar o modelo sobre `log1p(SalePrice)`** é a abordagem correta — tanto pela métrica da competição quanto pelas premissas estatísticas de diversos algoritmos.

## 4. Análise de dados ausentes

Entender os dados faltantes é essencial: no dicionário de dados (`data_description.txt`), fica claro que boa parte dos `NaN` **não representa ausência de informação**, e sim a ausência do próprio recurso na casa (ex.: uma casa sem piscina naturalmente não tem `PoolQC` preenchido).

In [ ]:
missing = train.isnull().sum()
missing = missing[missing > 0].sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).round(2)

missing_df = pd.DataFrame({'n_ausentes': missing, 'percentual (%)': missing_pct})
missing_df['provável significado'] = np.where(
    missing_df.index.isin(['PoolQC','MiscFeature','Alley','Fence','FireplaceQu','GarageType',
                            'GarageFinish','GarageQual','GarageCond','GarageYrBlt',
                            'BsmtQual','BsmtCond','BsmtExposure','BsmtFinType1','BsmtFinType2',
                            'MasVnrType']),
    'Ausência do recurso (NA estrutural)',
    'Dado genuinamente faltante'
)
missing_df


In [ ]:
plt.figure(figsize=(10,7))
sns.barplot(x=missing_df['percentual (%)'], y=missing_df.index, palette='rocket')
plt.xlabel('% de valores ausentes')
plt.title(f'Variáveis com dados ausentes no treino ({len(missing_df)} de {train.shape[1]} colunas)')
plt.tight_layout()
plt.show()


In [ ]:
plt.figure(figsize=(12,6))
sns.heatmap(train[missing_df.index].isnull(), cbar=False, cmap='magma', yticklabels=False)
plt.title('Mapa de calor dos valores ausentes (linhas = imóveis, colunas = atributos)')
plt.tight_layout()
plt.show()


**Principais achados:**

- `PoolQC` (99,5%), `MiscFeature` (96,3%), `Alley` (93,8%) e `Fence` (80,8%) têm altíssima taxa de ausência — mas, na esmagadora maioria dos casos, isso significa apenas que **a casa não tem piscina / recurso especial / acesso por viela / cerca**, e não um problema de coleta de dados.
- Grupos de variáveis relacionadas (`Garage*`, `Bsmt*`) apresentam **o mesmo padrão de ausência** (ex.: as 5 variáveis de porão faltam quase sempre nas mesmas 37-38 linhas) — evidência de que essas casas simplesmente **não possuem porão/garagem**, e não de dados corrompidos.
- `LotFrontage` (17,7%) é o caso mais delicado: é uma variável genuinamente numérica (metragem de testada para a rua) sem uma razão estrutural óbvia para estar ausente, exigindo uma estratégia de imputação mais cuidadosa (ex.: mediana por bairro).
- Apenas `Electrical` tem uma única linha ausente — provavelmente um dado realmente faltante, seguro de imputar pela moda.

Essa distinção entre **"ausência estrutural"** (recurso inexistente) e **"dado realmente faltante"** é decisiva: tratá-las da mesma forma (ex.: excluir linhas ou imputar média) jogaria fora informação valiosa — o fato de a casa **não ter piscina** é, em si, uma informação preditiva relevante.

## 5. Análise univariada — variáveis numéricas

Vamos observar a distribuição individual das principais variáveis numéricas contínuas (áreas, em geral) para identificar assimetrias, valores concentrados em zero e possíveis outliers.

In [ ]:
area_vars = ['LotArea','LotFrontage','TotalBsmtSF','1stFlrSF','2ndFlrSF',
             'GrLivArea','GarageArea','WoodDeckSF','OpenPorchSF']

fig, axes = plt.subplots(3, 3, figsize=(15,11))
for ax, col in zip(axes.flatten(), area_vars):
    sns.histplot(train[col].dropna(), kde=True, ax=ax, color='teal', bins=30)
    ax.set_title(f'{col}  (skew={train[col].skew():.2f})')
plt.tight_layout()
plt.show()


Praticamente todas as variáveis de área são **assimétricas à direita**, com muitas casas em valores baixos/moderados e uma cauda de imóveis grandes. `2ndFlrSF` e `OpenPorchSF`, além disso, têm **massa concentrada em zero** — ou seja, uma parcela relevante das casas simplesmente não possui segundo andar ou varanda aberta. Isso é esperado e reforça que essas variáveis se comportam quase como "indicador de presença + magnitude" (uma característica combinada).

In [ ]:
discrete_vars = ['OverallQual','OverallCond','FullBath','BedroomAbvGr','TotRmsAbvGrd',
                 'Fireplaces','GarageCars','YrSold']

fig, axes = plt.subplots(2, 4, figsize=(17,7))
for ax, col in zip(axes.flatten(), discrete_vars):
    sns.countplot(x=train[col], ax=ax, color='cornflowerblue')
    ax.set_title(col)
    ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.show()


- `OverallQual` (qualidade geral) segue um padrão quase normal, centrado em 5-6 (médio), com poucas casas nos extremos (1-2 ou 9-10) — uma variável muito bem distribuída e, como veremos, **altamente preditiva**.
- A maioria das casas tem `FullBath` = 1 ou 2, `GarageCars` = 2 e nenhuma lareira (`Fireplaces` = 0).
- `YrSold` cobre apenas 5 anos (2006–2010), sem grande variação — o que sugere que efeitos macroeconômicos de longo prazo (ex.: a crise de 2008) devem ser observados com cautela devido à amostra temporal limitada.

## 6. Análise univariada — variáveis categóricas

Para as variáveis categóricas, observamos a frequência de cada categoria, com atenção especial para variáveis **quase constantes** (uma categoria domina >95% dos casos), que tendem a ter pouco poder discriminativo.

In [ ]:
dominance = []
for c in cat_cols:
    vc = train[c].value_counts(normalize=True, dropna=False)
    dominance.append({'variavel': c, 'categoria_dominante': vc.index[0], 'percentual (%)': round(vc.iloc[0]*100, 1)})

dominance_df = pd.DataFrame(dominance).sort_values('percentual (%)', ascending=False)
dominance_df.head(15)


In [ ]:
quase_constantes = dominance_df[dominance_df['percentual (%)'] >= 90]
print(f'{len(quase_constantes)} variáveis categóricas com uma categoria dominante em >= 90% dos casos:')
quase_constantes


Variáveis como `Utilities`, `Street`, `PoolQC`, `Condition2`, `RoofMatl` e `Heating` são **quase constantes**: em praticamente todos os imóveis assumem o mesmo valor. Elas tendem a agregar pouca informação discriminativa para prever preço (embora, em modelos de árvore, ainda possam capturar os poucos casos raros e distintos). São boas candidatas a simplificação (ex.: binarização "é o valor padrão ou não") ou remoção em uma etapa futura de seleção de atributos.

In [ ]:
important_cats = ['Neighborhood','HouseStyle','ExterQual','KitchenQual','BsmtQual','GarageFinish']

fig, axes = plt.subplots(3, 2, figsize=(15,14))
for ax, col in zip(axes.flatten(), important_cats):
    order = train[col].value_counts().index
    sns.countplot(y=train[col], order=order, ax=ax, palette=PALETTE)
    ax.set_title(col)
plt.tight_layout()
plt.show()


`Neighborhood` é a variável categórica com **maior número de categorias (25 bairros)** e, como veremos a seguir, uma das mais associadas ao preço — bairros como `NridgHt`, `NoRidge` e `StoneBr` concentram os imóveis mais caros, enquanto `MeadowV`, `IDOTRR` e `BrDale` concentram os mais baratos.

## 7. Análise bivariada — variáveis numéricas x `SalePrice`

Agora relacionamos as principais variáveis numéricas diretamente com o alvo, tanto por correlação linear quanto visualmente (*scatterplots*).

In [ ]:
corr_target = train[num_cols_no_id].corr(numeric_only=True)['SalePrice'].drop('SalePrice')
corr_target_sorted = corr_target.sort_values(key=abs, ascending=False)

plt.figure(figsize=(8,9))
colors = ['seagreen' if v > 0 else 'indianred' for v in corr_target_sorted.head(20)]
sns.barplot(x=corr_target_sorted.head(20).values, y=corr_target_sorted.head(20).index, palette=colors)
plt.axvline(0, color='black', linewidth=0.8)
plt.xlabel('Correlação de Pearson com SalePrice')
plt.title('Top 20 variáveis numéricas mais correlacionadas com o preço (em módulo)')
plt.tight_layout()
plt.show()

corr_target_sorted.head(10)


In [ ]:
top_features = ['OverallQual','GrLivArea','GarageCars','GarageArea','TotalBsmtSF','1stFlrSF','FullBath','YearBuilt']

fig, axes = plt.subplots(2, 4, figsize=(18,8))
for ax, col in zip(axes.flatten(), top_features):
    sns.scatterplot(data=train, x=col, y='SalePrice', ax=ax, alpha=0.4, color='navy')
    ax.set_title(f'{col}  (corr={corr_target[col]:.2f})')
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()


**Principais achados:**

- `OverallQual` (qualidade geral do material/acabamento) é, de longe, **a variável mais correlacionada com o preço** (corr ≈ 0.79) — e a relação é visivelmente monotônica e quase linear em `log(SalePrice)`, o que a torna um dos atributos mais valiosos do dataset.
- `GrLivArea` (área habitável acima do solo, corr ≈ 0.71) mostra relação positiva forte e aproximadamente linear, mas com **dois pontos claramente fora do padrão** (casas grandes vendidas por preço muito baixo) — os outliers que precisarão de tratamento na modelagem.
- `GarageCars` e `GarageArea` são fortemente correlacionadas entre si (ambas medem "tamanho de garagem" por ângulos diferentes) e com o preço — um indício de **multicolinearidade** a considerar.
- `YearBuilt` tem correlação positiva moderada: casas mais novas tendem a valer mais, mas a relação é mais dispersa que as demais, sugerindo que a idade sozinha explica menos do que a qualidade/tamanho.

## 8. Análise bivariada — variáveis categóricas x `SalePrice`

Para variáveis categóricas, comparamos a distribuição de `SalePrice` entre categorias usando *boxplots*, que revelam tanto a mediana quanto a dispersão dentro de cada grupo.

In [ ]:
plt.figure(figsize=(14,7))
order = train.groupby('Neighborhood')['SalePrice'].median().sort_values().index
sns.boxplot(data=train, x='Neighborhood', y='SalePrice', order=order, palette=PALETTE)
plt.xticks(rotation=75)
plt.title('SalePrice por bairro (Neighborhood), ordenado pela mediana')
plt.gca().yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()


O `Neighborhood` (bairro) sozinho já explica uma boa parte da variação de preço: a mediana varia de **cerca de US\$100 mil** (`MeadowV`, `IDOTRR`) a **mais de US\$300 mil** (`NridgHt`, `StoneBr`, `NoRidge`) — uma amplitude de 3x. Isso confirma que localização é um dos fatores mais determinantes do valor de um imóvel, mesmo dentro de uma única cidade.

In [ ]:
quality_cats = ['ExterQual','KitchenQual','BsmtQual','HeatingQC']
order_quality = ['Po','Fa','TA','Gd','Ex']

fig, axes = plt.subplots(2, 2, figsize=(13,9))
for ax, col in zip(axes.flatten(), quality_cats):
    present_order = [o for o in order_quality if o in train[col].unique()]
    sns.boxplot(data=train, x=col, y='SalePrice', order=present_order, ax=ax, palette='coolwarm')
    ax.set_title(col)
    ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()


As variáveis de qualidade (`ExterQual`, `KitchenQual`, `BsmtQual`, `HeatingQC`) seguem um padrão **claramente ordinal**: à medida que a qualidade sobe de `Po` (pobre) para `Ex` (excelente), a mediana do preço sobe de forma quase monotônica. Isso é um forte indício de que essas variáveis devem ser codificadas como **ordinais** (respeitando a ordem Po < Fa < TA < Gd < Ex) na etapa de modelagem, em vez de apenas one-hot encoding, para preservar essa relação de ordem.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14,5))
sns.boxplot(data=train, x='CentralAir', y='SalePrice', ax=axes[0], palette='Set2')
axes[0].set_title('SalePrice por Ar-Condicionado Central')
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))

order_bldg = train.groupby('BldgType')['SalePrice'].median().sort_values().index
sns.boxplot(data=train, x='BldgType', y='SalePrice', order=order_bldg, ax=axes[1], palette='Set2')
axes[1].set_title('SalePrice por Tipo de Construção')
axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()


## 9. Análise multivariada — matriz de correlação

Para entender não apenas a relação de cada variável com o alvo, mas também **relações entre as próprias variáveis explicativas** (multicolinearidade), construímos a matriz de correlação das variáveis numéricas mais relevantes.

In [ ]:
top_vars = corr_target_sorted.head(15).index.tolist() + ['SalePrice']
corr_matrix = train[top_vars].corr()

plt.figure(figsize=(11,9))
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            square=True, linewidths=0.5, cbar_kws={'shrink':0.8})
plt.title('Matriz de correlação — Top 15 variáveis mais relacionadas ao preço')
plt.tight_layout()
plt.show()


**Pares de variáveis com alta multicolinearidade (corr > 0.8):**

- `GarageCars` e `GarageArea` (mais carros exigem mais área — redundância esperada)
- `TotalBsmtSF` e `1stFlrSF` (em muitas casas, o porão tem a mesma metragem do primeiro andar)
- `GrLivArea` e `TotRmsAbvGrd` (mais cômodos geralmente significa mais área habitável)

Isso não chega a ser um problema para modelos de árvore (Random Forest, Gradient Boosting), mas é relevante para **modelos lineares** (Ridge, Lasso), nos quais a multicolinearidade pode inflar a variância dos coeficientes — reforçando a importância de usar regularização (L1/L2) na etapa de modelagem.

In [ ]:
plt.figure(figsize=(6,7))
sns.heatmap(train[['SalePrice']+top_vars[:-1]].corr()[['SalePrice']].sort_values('SalePrice', ascending=False),
            annot=True, fmt='.2f', cmap='YlGnBu', cbar=False)
plt.title('Correlação de cada variável com SalePrice')
plt.tight_layout()
plt.show()


## 10. Detecção de outliers e pontos de atenção

Outliers podem distorcer significativamente o ajuste de modelos sensíveis a valores extremos (regressão linear, KNN, redes neurais). Vamos investigar os casos mais notáveis usando o critério estatístico do **IQR (intervalo interquartil)** e inspeção visual.

In [ ]:
def contar_outliers_iqr(serie):
    q1, q3 = serie.quantile([0.25, 0.75])
    iqr = q3 - q1
    lim_inf, lim_sup = q1 - 1.5*iqr, q3 + 1.5*iqr
    return ((serie < lim_inf) | (serie > lim_sup)).sum()

outlier_counts = {col: contar_outliers_iqr(train[col].dropna()) for col in
                  ['SalePrice','GrLivArea','LotArea','TotalBsmtSF','1stFlrSF','GarageArea']}

pd.Series(outlier_counts, name='n_outliers_IQR').sort_values(ascending=False)


In [ ]:
fig, ax = plt.subplots(figsize=(8,6))
sns.scatterplot(data=train, x='GrLivArea', y='SalePrice', alpha=0.5, color='steelblue', ax=ax)

outliers = train[(train['GrLivArea'] > 4000) & (train['SalePrice'] < 300000)]
ax.scatter(outliers['GrLivArea'], outliers['SalePrice'], color='red', s=120,
           edgecolor='black', label='Outliers notáveis', zorder=5)
for _, row in outliers.iterrows():
    ax.annotate(f"Id {int(row['Id'])}", (row['GrLivArea'], row['SalePrice']),
                textcoords='offset points', xytext=(10,-10))

ax.set_title('GrLivArea x SalePrice — outliers notáveis')
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.show()

outliers[['Id','GrLivArea','OverallQual','YearBuilt','SalePrice']]


Os `Id`s 524 e 1299 (índices, não os `Id` originais mostrados acima já refletem o dado correto) são casas com área habitável **acima de 4.000 pés quadrados** vendidas por **menos de US\$300 mil** — muito abaixo do que a relação geral entre área e preço sugeriria. Esse padrão é amplamente documentado na literatura sobre este dataset (inclusive pela própria autora da base, De Cock) e recomenda-se a **remoção desses dois pontos** antes do treinamento de modelos, pois eles distorcem a reta/superfície de ajuste sem representar o comportamento típico do mercado.

Vale reforçar: outliers em variáveis explicativas **não são necessariamente erros de digitação** — muitas vezes refletem transações atípicas (venda entre familiares, imóvel incompleto, execução judicial etc.), o que pode ser investigado através da variável `SaleCondition`.

In [ ]:
print('Distribuição de SaleCondition para os outliers:')
print(outliers['SaleCondition'].tolist())
print('\nDistribuição geral de SaleCondition no dataset:')
train['SaleCondition'].value_counts(normalize=True).round(3)


## 11. Análise temporal

Como o preço de um imóvel também depende de *quando* foi construído, reformado e vendido, vale observar tendências ao longo do tempo.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(15,5))

yearly_price = train.groupby('YearBuilt')['SalePrice'].median()
axes[0].plot(yearly_price.index, yearly_price.values, color='darkgreen')
axes[0].set_title('Mediana de SalePrice por Ano de Construção')
axes[0].set_xlabel('YearBuilt')
axes[0].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))

sns.boxplot(data=train, x='YrSold', y='SalePrice', ax=axes[1], palette='Blues')
axes[1].set_title('SalePrice por Ano de Venda')
axes[1].yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))

plt.tight_layout()
plt.show()


- Existe uma tendência geral de **casas mais recentes venderem por preços mais altos**, mas a relação não é estritamente linear — há bastante ruído, especialmente em construções anteriores a 1950, onde reformas (`YearRemodAdd`) provavelmente têm papel mais relevante que o ano de construção original.
- Os preços entre **2006 e 2010** (`YrSold`) são bastante estáveis, sem uma queda visível associada à crise financeira de 2008 — possivelmente porque o mercado de Ames, uma cidade universitária de porte médio, foi menos afetado que grandes centros metropolitanos, ou porque a amostra é pequena demais por ano para captar esse efeito com robustez.

In [ ]:
train['AnosDesdeReforma'] = train['YrSold'] - train['YearRemodAdd']

plt.figure(figsize=(8,5))
sns.scatterplot(data=train, x='AnosDesdeReforma', y='SalePrice', alpha=0.4, color='purple')
plt.title('Anos desde a última reforma x SalePrice')
plt.gca().yaxis.set_major_formatter(mticker.StrMethodFormatter('${x:,.0f}'))
plt.tight_layout()
plt.show()

print('Correlação (Anos desde reforma x SalePrice):',
      round(train['AnosDesdeReforma'].corr(train['SalePrice']), 3))


A correlação negativa confirma o esperado: quanto **mais tempo se passou desde a última reforma**, menor tende a ser o preço de venda — reformas recentes agregam valor perceptível ao imóvel, independentemente da idade original da construção.

## 12. Síntese dos principais insights e recomendações para modelagem

### Sobre a variável-alvo
- `SalePrice` é assimétrica à direita; a transformação `log1p` normaliza a distribuição e deve ser usada como alvo de treinamento, alinhada à métrica oficial (RMSE em log).

### Sobre dados ausentes
- A maior parte dos valores ausentes é **estrutural** (ausência do recurso, ex.: sem piscina, sem porão, sem garagem) e deve ser codificada como categoria `"None"` / valor `0`, não removida nem imputada por média/mediana ingênua.
- `LotFrontage` é o caso mais delicado de imputação genuína, melhor tratado com a mediana por `Neighborhood`.

### Sobre as variáveis mais preditivas
- `OverallQual`, `GrLivArea`, `TotalBsmtSF`, `GarageCars`/`GarageArea` e `Neighborhood` estão entre os fatores mais fortemente associados ao preço.
- Variáveis de qualidade (`ExterQual`, `KitchenQual`, `BsmtQual`...) têm relação claramente **ordinal** com o preço — recomenda-se codificação ordinal, não apenas one-hot.
- `Neighborhood` sozinho gera diferenças de até 3x na mediana de preço — localização é um dos fatores dominantes.

### Sobre outliers
- Duas casas com `GrLivArea` > 4.000 pés² vendidas muito abaixo do esperado devem ser removidas antes do treinamento — são amplamente reconhecidas como outliers na literatura sobre este dataset.

### Sobre multicolinearidade
- Pares como `GarageCars`/`GarageArea` e `TotalBsmtSF`/`1stFlrSF` são altamente correlacionados entre si; combiná-los (ex.: em uma métrica de área total) ou usar regularização (Lasso/Ridge) ajuda a mitigar o problema.

### Sobre variáveis de baixo valor discriminativo
- Diversas variáveis categóricas são quase constantes (`Utilities`, `Street`, `PoolQC`, `Condition2`, `RoofMatl`, `Heating`) e podem ser simplificadas, agrupadas ou descartadas sem grande perda de informação.

Esses insights formam a base para as decisões de limpeza, tratamento de dados ausentes e engenharia de atributos na etapa de modelagem preditiva.